# Modes dynamiques de la vidéo — MODICA GRAZIANA OD

**Physics-informed DMD** (`pydmd.PiDMD`) sur les vidéos recalées `cascade_v9` des trois réplicats
OD1 / OD2 / OD3, côte à côte. Le carnet n'écrit rien sur le disque.

**Noyau : l'environnement conda `pyOR`** (c'est lui qui a `pydmd`).

## Ce que la chaîne fait

1. la vidéo recalée (`RawImages/registered_cascade_v9_1536x1024/registered_video.mp4`) et ses masques
   sont relus tels quels — aucun re-recalage ;
2. une **ROI 2D** est découpée autour de la choroïde (union des masques ± marges, fraction de colonnes),
   puis moyennée par blocs `DS_Y × DS_X` ; chaque image devient un vecteur d'état, **aplati en ordre
   Fortran** (c'est l'ordre qu'attend PyDMD pour les variétés `BC*`, cf. `pidmd_utils.fft_block2d`) ;
3. `PiDMD(manifold=…)` ajuste l'opérateur `A` de `X[:, t+1] = A X[:, t]`, **restreint à une famille de
   matrices** ; les modes propres de `A` sont les modes spatiaux, `b_k λ_k^t` les modes temporels.

`MANIFOLD` accepte **une variété ou plusieurs** (`("unitary", "circulant")`) : chaque variété est
ajustée sur chaque réplicat, et les figures les comparent.

## Les autres méthodes : BOP-DMD et COSTS

`METHODE` choisit **une seule** des trois méthodes — `"pi-DMD"` (les variétés de `MANIFOLD`),
`"BOP-DMD"`, ou `"COSTS"` — et les blocs de paramètres des autres sont ignorés. Les deux premières
partagent les mêmes figures (§3) ; `"COSTS"` a les siennes (§4).

**BOP-DMD** (*bagging, optimized DMD* —
Sashidhar & Kutz 2022, Phil. Trans. R. Soc. A 380:20210199). Il ne contraint pas la forme de `A` : il
ajuste directement des exponentielles `exp(λ t)` **sur tous les instants à la fois** (projection
variable), puis recommence sur des **tirages aléatoires de snapshots** (*bagging*) et rend la moyenne
et l'**écart-type** des valeurs propres, des modes et des amplitudes. Trois différences avec les
`PiDMD` :

- ses valeurs propres sont en **temps continu** : `fréquence = Im(λ)/2π`, croissance `= Re(λ)` — surtout
  pas `log(λ)/dt`. Le carnet en tient compte et trace `exp(λ dt)` là où il faut comparer au cercle unité ;
- il accepte l'**échantillonnage irrégulier** : avec `BOP_TEMPS_REELS`, il travaille sur les horodatages
  bruts, sans l'interpolation dont les autres méthodes ont besoin ;
- le bagging donne des **barres d'erreur**, reportées sur les figures.

Ce qu'il donne ici, et qu'aucune variété pi-DMD n'avait sorti : **la fondamentale cardiaque et ses
harmoniques**, 60,8 / 60,8 / 60,4 BPM sur OD1 / OD2 / OD3 pour une FC de 60,7, puis ≈ 120, 183 et
242 BPM. Quatre choses à savoir avant de toucher aux réglages :

- `init_alpha` est **décisif** : sans lui, BOP-DMD se cale sur la dérive (1,5 BPM). D'où `BOP_INIT_FC` ;
- le bagging bouge **l'amortissement, pas la fréquence** : l'écart-type sur `Im(λ)` est sous 0,01 BPM
  alors que celui sur `Re(λ)` va jusqu'à ~1 s⁻¹. Les barres d'erreur sont donc verticales ;
- le bagging **échoue parfois** (`LinAlgError`, y compris avec `remove_bad_bags`) : repli automatique
  sur `num_trials = 0`, signalé dans le journal. Rang 30 + bagging échoue toujours. Les tirages passent
  par le RNG global de numpy, d'où `BOP_SEED` — sans graine, deux exécutions ne donnent pas la même
  fréquence (56 ou 61 BPM sur OD2 d'une fois à l'autre) ;
- coût : 15 à 30 s par réplicat. `BOP_NUM_TRIALS = 0` ramène à ~2 s, sans les barres d'erreur.

## Et si la fréquence n'est pas constante : COSTS

Un mode DMD **est** `b·exp(λt)` : sa fréquence est constante par construction. Une FC qui dérive est
donc mal représentée — l'ajustement l'étale sur plusieurs modes voisins, ou amortit la paire pour
rattraper le déphasage accumulé. `METHODE = "COSTS"` (*Coherent Spatio-Temporal Scale separation*,
Dylewsky, Tao & Kutz, Phys. Rev. E 99:063311, 2019 ; `pydmd.costs.COSTS`) fait glisser une fenêtre
courte sur la vidéo, ajuste un BOP-DMD **dans chaque fenêtre**, puis regroupe les valeurs propres en
bandes de fréquence par *k*-means. On lit alors **f(t)**, pas un `f` unique.

Le réglage qui compte est `COSTS_WINDOW` : elle doit tenir ~3 à 5 cycles (48 frames ≈ 5 s ≈ 5 battements
à 60 BPM), et la résolution en fréquence vaut à peu près l'inverse de sa durée — fenêtre courte, meilleur
suivi de la dérive mais séparation plus floue entre la fondamentale et ses harmoniques. Mesuré ici :
26 fenêtres, **75 s par réplicat**, et une FC qui va de 55 à 67 BPM au fil de l'acquisition (médiane
62,4, FC de référence 60,7). Deux détails d'API : `fit` veut le temps en **2D** (`t[None, :]`), et son
`omega` est en temps continu comme BOP-DMD.

## Les variétés disponibles

Les 19 noms acceptés par PyDMD, **sensibles à la casse** (attention : `circulant_unitary` avec un
tiret bas, mais `BCCBunitary` sans) :

| famille | `manifold` | lecture physique ici |
|---|---|---|
| conservation / symétrie | `unitary`, `symmetric`, `skewsymmetric` | énergie conservée, dynamique auto-adjointe, purement oscillante |
| localité | `diagonal` (`manifold_opt` = largeur de bande), `symmetric_tridiagonal` | un pixel n'est couplé qu'à ses voisins |
| causalité | `uppertriangular`, `lowertriangular` | propagation dans un seul sens |
| invariance par translation | `toeplitz`, `hankel`, `circulant`, `circulant_unitary`, `circulant_symmetric`, `circulant_skewsymmetric` | milieu homogène (le `circulant*` impose en plus la périodicité) |
| blocs (2D) | `BC`, `BCTB`, `BCCB`, `BCCBunitary`, `BCCBsymmetric`, `BCCBskewsymmetric` | `manifold_opt = (h, w)` : **circulant d'une colonne A-scan à l'autre**, couplage libre (`BC`), tridiagonal (`BCTB`) ou circulant (`BCCB`) **en profondeur** |

`manifold_opt` ne sert qu'à `diagonal` (entier `k` → bande `k-1` de part et d'autre ; couple
`(k1, k2)` → bandes basse et haute ; tableau `(n, 2)` → bornes explicites) et aux `BC*`
(dimensions des blocs). Partout ailleurs il est ignoré — le carnet le met à `None`.

**Trois pièges vérifiés dans le source de PyDMD 2025.8.1**, dont le carnet tient compte :

- les variétés `uppertriangular`, `lowertriangular`, `toeplitz`, `hankel` et toutes les `BC*`
  **exigent `compute_A=True`** (sinon `ValueError: A must be computed…`). `A` est alors une matrice
  `n × n` **pleine**, diagonalisée en entier : `svd_rank` ne tronque plus rien et on récupère `n` modes.
  D'où le garde-fou mémoire et une ROI volontairement sous-échantillonnée ;
- avec ces mêmes variétés, `opt=True` casse le calcul des amplitudes (les modes ne sont plus au rang
  SVD) : le carnet repasse alors automatiquement à `opt=False` ;
- l'aplatissement des `BC*` est en **ordre Fortran** sur `(h, w)`.


In [13]:
from __future__ import annotations

import difflib
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from pydmd import BOPDMD, DMD, PiDMD
from pydmd.costs import COSTS

# Racine du depot : le carnet vit dans notebook/ (``__file__`` n'existe pas dans un carnet).
REPO = Path.cwd().resolve()
if REPO.name == "notebook":
    REPO = REPO.parent
for p in (REPO / "src", REPO / "Astronauts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from ocularrigidity.data.compression import read_gray  # noqa: E402
from ocularrigidity.data.io import load_mask  # noqa: E402

pio.templates.default = "plotly_white"
warnings.filterwarnings("ignore", message=".*ill-conditioned.*")
print(f"depot : {REPO}")

depot : C:\Users\transformer\Documents\Nicolas\mes-codes\SANSORI\OcularRigidity


In [14]:
# --------------------------------------------------------------------------- #
# Parametres — TOUT ce qui se regle a la main est ici
# --------------------------------------------------------------------------- #
PATH_GENERAL = Path("E:/SANSORI/Reproducibility")
SEGVAR = Path("E:/NASA_Rigidity/Reproducibility/SegmentationVariations")
DATA_SUBDIR = "registered_cascade_v9_1536x1024"
CONDITIONS = SEGVAR / "cascade_v9_1536x1024_hr_mediane" / "one_cycle_6beats_6bins" / "conditions.csv"
SLUGS = ("MODICA_GRAZIANA_OD1", "MODICA_GRAZIANA_OD2", "MODICA_GRAZIANA_OD3")

# --- Affichage de la video ---------------------------------------------------
VIDEO_STEP = 3          # 1 frame sur n dans l'animation (c'est elle qui pese : ~15 Mo a 3)
VIDEO_DS = 2            # sous-echantillonnage spatial de l'animation
CONTOUR_MASQUE = True   # surligner le contour du masque choroide

# --- ROI et pretraitement ----------------------------------------------------
MARGE_HAUT_PX = 75      # au-dessus du masque (RPE / retine externe)
MARGE_BAS_PX = 15       # au-dessous (sclere)
COL_FRAC = (0.4, 0.6)   # fraction de largeur gardee (bords = artefacts de recalage)
DS_Y, DS_X = 1, 3       # moyenne par blocs : ROI de ~115 x 576 -> ~29 x 72 = ~2000 etats
NORMALISER_PAR_FRAME = True   # divise chaque frame par sa moyenne ROI (derive d'illumination)
SOUSTRAIRE_MOYENNE = True     # retire la moyenne temporelle pixel par pixel
REECHANTILLONNER_UNIFORME = True  # interpole sur une grille de pas dt median

# --- Quelle methode ? --------------------------------------------------------
# "pi-DMD" : les varietes de MANIFOLD ci-dessous.
# "BOP-DMD" : le bloc BOP_* plus bas.
# "COSTS"  : BOP-DMD sur fenetre glissante -> f(t), pour une frequence qui derive.
# Une seule tourne ; les blocs de parametres des autres sont ignores.
METHODE = "COSTS"

# --- pi-DMD ------------------------------------------------------------------
# UNE variete ("unitary") ou PLUSIEURS ("unitary", "circulant", ...) : chacune est ajustee
# sur chaque replicat et les figures les comparent. Noms exacts : voir l'introduction.
MANIFOLD = ("unitary", "circulant", "diagonal")
# Valeur unique appliquee a toutes, ou dict {variete: option}. Ignore hors "diagonal" et "BC*".
MANIFOLD_OPT = {"diagonal": (10, 10)}   # "BC*" -> None = (h, w) de la ROI
SVD_RANK = 30           # rang de troncature (ignore par les varietes a A plein, voir l'introduction)
TLSQ_RANK = 0
COMPUTE_A = False       # force a True pour les varietes qui l'exigent
OPT = True              # amplitudes facon "optimized DMD" (desactive si compute_A)
MEMOIRE_MAX_GO = 4.0    # garde-fou : refuse de former un A de n x n complexes plus gros

COMPARER_DMD_STANDARD = True   # meme rang, sans contrainte, comme reference

# --- BOP-DMD (Sashidhar & Kutz 2022) ----------------------------------------
BOP_RANK = 12                # rang : 30 + bagging echoue sur ces donnees
BOP_INIT_FC = True           # init_alpha = FC du replicat et ses harmoniques (decisif)
BOP_NUM_TRIALS = 20          # tirages du bagging (0 = optimized DMD simple) ; ~16 s / replicat
BOP_TRIAL_SIZE = 0.6         # fraction de snapshots par tirage
BOP_EIG_CONSTRAINTS = {"conjugate_pairs"}   # aussi : "stable", "imag"
BOP_MAXITER = 200            # iterations de la projection variable
BOP_TEMPS_REELS = True       # ajuster sur les horodatages bruts (BOP-DMD n'exige pas dt constant)
BOP_SEED = 0                 # graine des tirages : sans elle, deux executions ne donnent pas la meme FC

# --- COSTS (BOP-DMD sur fenetre glissante) ----------------------------------
COSTS_RANK = 8               # rang BOP-DMD DANS chaque fenetre
COSTS_WINDOW = 48            # longueur de fenetre, en frames (48 ~ 5 s ~ 5 battements a 60 BPM)
COSTS_STEP = 12              # pas de glissement, en frames (6 -> 51 fenetres, beaucoup plus lent)
COSTS_N_BANDES = 3           # bandes de frequence : k-means sur Im(omega)
COSTS_TRANSFORM = "log10"    # "absolute" | "log10" | "period" | "square_frequencies"
COSTS_MAXITER = 30           # iterations de la projection variable, par fenetre
COSTS_GLOBAL_SVD = True      # base POD commune a toutes les fenetres (plus rapide)
COSTS_USE_LAST_FREQ = False  # initialiser chaque fenetre sur les eigs de la precedente
COSTS_N_CARTES = 4           # nombre de fenetres montrees sur les cartes spatiales

# --- Modes affiches ----------------------------------------------------------
N_MODES = 4                  # nombre de modes montres par (variete, replicat)
TRI_MODES = "proche_fc"      # "amplitude" | "proche_fc"
BANDE_FC_FRAC = 0.2          # bande cardiaque affichee : FC +/- 20 %
AFFICHER_PHASE = True        # seconde figure de modes spatiaux : la phase

# --- Constantes de PyDMD (verifiees dans pydmd/pidmd.py) ---------------------
MANIFOLDS_BLOCS = {"BC", "BCTB", "BCCB", "BCCBunitary", "BCCBsymmetric", "BCCBskewsymmetric"}
MANIFOLDS_A_PLEIN = {"uppertriangular", "lowertriangular", "toeplitz", "hankel"} | MANIFOLDS_BLOCS
MANIFOLDS_OPT_UTILE = {"diagonal"} | MANIFOLDS_BLOCS
MANIFOLDS_VALIDES = {
    "unitary", "uppertriangular", "lowertriangular", "diagonal", "symmetric", "skewsymmetric",
    "toeplitz", "hankel", "circulant", "circulant_unitary", "circulant_symmetric",
    "circulant_skewsymmetric", "symmetric_tridiagonal",
} | MANIFOLDS_BLOCS

BOP = "BOP-DMD"
METHODES = ("pi-DMD", BOP, "COSTS")
if METHODE not in METHODES:
    raise ValueError(f"METHODE doit etre l'une de {METHODES}, pas {METHODE!r}")
EST_COSTS = METHODE == "COSTS"

MANIFOLDS = (MANIFOLD,) if isinstance(MANIFOLD, str) else tuple(MANIFOLD)
_inconnus = [m for m in MANIFOLDS if m not in MANIFOLDS_VALIDES] if METHODE == "pi-DMD" else []
if _inconnus:
    _proches = {m: difflib.get_close_matches(m, sorted(MANIFOLDS_VALIDES), n=2) for m in _inconnus}
    raise ValueError(f"variété(s) inconnue(s) : {_inconnus}. Proches : {_proches}. "
                     f"Noms acceptés : {sorted(MANIFOLDS_VALIDES)}")

# Les figures de la section 3 parcourent MODELES : les varietes de MANIFOLD, ou le seul
# BOP-DMD. En mode COSTS il est VIDE — cette methode a ses propres figures, section 4.
MODELES = () if EST_COSTS else (MANIFOLDS if METHODE == "pi-DMD" else (BOP,))

C_REPLICAT = {1: "#3987e5", 2: "#eb6834", 3: "#1baf7a"}
SYMB_REPLICAT = {1: "circle", 2: "square", 3: "diamond"}
C_MODELE = dict(zip(MODELES, px.colors.qualitative.D3))


def fr(x, chiffres=1):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "—"
    return f"{{:.{chiffres}f}}".format(x).replace(".", ",")


def replicat(slug):
    return int(slug[-1])

In [15]:
# --------------------------------------------------------------------------- #
# Chargement — frames, masques, horodatages, FC de reference
# --------------------------------------------------------------------------- #
conds = pd.read_csv(CONDITIONS).set_index("slug")


def charger(slug: str) -> dict:
    d = PATH_GENERAL / slug / "RawImages" / DATA_SUBDIR
    frames = read_gray(str(d / "registered_video.mp4"))            # (T, H, W) uint8
    masks = np.asarray(load_mask(d / "mask.npz"), dtype=bool)      # (T, H, W)
    ts_us = np.loadtxt(d / "timestamp.txt")                        # microsecondes
    if not (len(frames) == len(masks) == len(ts_us)):
        raise ValueError(f"{slug}: {len(frames)} frames, {len(masks)} masques, {len(ts_us)} horodatages")
    t = (ts_us - ts_us[0]) / 1e6
    dt = float(np.median(np.diff(t)))
    return {
        "slug": slug, "replicat": replicat(slug), "frames": frames, "masks": masks,
        "t": t, "dt": dt, "fs": 1.0 / dt,
        "hr_bpm": float(conds.loc[slug, "hr_BPM"]) if slug in conds.index else np.nan,
    }


t0 = time.perf_counter()
DATA = {s: charger(s) for s in SLUGS}
print(f"charge en {time.perf_counter() - t0:.1f} s")

display(pd.DataFrame([{
    "réplicat": d["replicat"], "frames": len(d["frames"]),
    "H × W": f"{d['frames'].shape[1]} × {d['frames'].shape[2]}",
    "dt médian (ms)": round(1000 * d["dt"], 1), "fs (Hz)": round(d["fs"], 2),
    "durée (s)": round(d["t"][-1], 1), "FC (BPM)": round(d["hr_bpm"], 1),
    "Nyquist (BPM)": round(30 * d["fs"], 0),
} for d in DATA.values()]).set_index("réplicat"))

charge en 1.4 s


,frames,H × W,dt médian (ms),fs (Hz),durée (s),FC (BPM),Nyquist (BPM)
réplicat,,,,,,,
1,348,496 × 768,105.0,9.52,37.1,60.7,286.0
2,348,496 × 768,104.0,9.62,37.2,60.7,288.0
3,348,496 × 768,104.0,9.62,36.8,60.7,288.0


## 1. La vidéo recalée

Les trois réplicats à la même cadence d'animation (`VIDEO_STEP` frames sur une). Le contour du masque
de choroïde est surligné si `CONTOUR_MASQUE`. L'animation est sous-échantillonnée : c'est elle qui pèse
dans la taille du carnet.

In [ ]:
def apercu(d):
    """Frames sous-echantillonnees, contour du masque surligne."""
    f = d["frames"][::VIDEO_STEP, ::VIDEO_DS, ::VIDEO_DS].astype(np.uint8)
    if CONTOUR_MASQUE:
        m = d["masks"][::VIDEO_STEP, ::VIDEO_DS, ::VIDEO_DS]
        bord = np.zeros_like(m)
        bord[:, 1:, :] |= m[:, 1:, :] != m[:, :-1, :]    # transitions verticales
        bord[:, :, 1:] |= m[:, :, 1:] != m[:, :, :-1]    # transitions horizontales
        f = np.where(bord, 255, f).astype(np.uint8)
    return f


n_t = min(len(apercu(DATA[s])) for s in SLUGS)
video = np.stack([apercu(DATA[s])[:n_t] for s in SLUGS], axis=-1)   # (T, H, W, replicat)

fig = px.imshow(video, animation_frame=0, facet_col=3, binary_string=True,
                facet_col_spacing=0.01,
                labels={"animation_frame": f"frame (× {VIDEO_STEP})"})
for i, s in enumerate(SLUGS):
    fig.layout.annotations[i]["text"] = f"OD{replicat(s)}"
fig.update_layout(height=340, margin=dict(l=10, r=10, t=40, b=10),
                  title="Vidéos recalées cascade_v9 — MODICA GRAZIANA OD")
fig.show()

## 2. La pi-DMD

La ROI est fixée par l'union des masques (± marges) et la fraction de colonnes, puis moyennée par blocs.
Le garde-fou mémoire s'applique aux variétés qui forment `A` en entier.

In [ ]:
def bloc_moyen(cube, dy, dx):
    """Moyenne par blocs dy x dx sur les deux dernieres dimensions (recadre au multiple)."""
    T, h, w = cube.shape
    h, w = (h // dy) * dy, (w // dx) * dx
    return cube[:, :h, :w].reshape(T, h // dy, dy, w // dx, dx).mean(axis=(2, 4))


def construire_X(d: dict) -> dict:
    """ROI 2D -> matrice d'etats X (n, T), aplatie en ordre FORTRAN (exige par les BC*)."""
    frames, masks, t = d["frames"], d["masks"], d["t"]
    H, W = frames.shape[1:]
    lignes = np.where(masks.any(axis=(0, 2)))[0]
    r0 = max(int(lignes.min()) - MARGE_HAUT_PX, 0)
    r1 = min(int(lignes.max()) + 1 + MARGE_BAS_PX, H)
    c0, c1 = int(COL_FRAC[0] * W), int(COL_FRAC[1] * W)

    roi = frames[:, r0:r1, c0:c1].astype(np.float64) / 255.0
    if NORMALISER_PAR_FRAME:
        moy = roi.mean(axis=(1, 2), keepdims=True)
        roi = roi / np.where(moy > 0, moy, 1.0) * roi.mean()
    roi = bloc_moyen(roi, DS_Y, DS_X)                       # (T, h, w)

    # (T, h, w) -> (n, T), ordre Fortran par image, moyenne temporelle retiree
    def en_etats(cube):
        n_t, hh, ww = cube.shape
        M = cube.reshape(n_t, hh * ww, order="F").T.copy()
        return (M - M.mean(axis=1, keepdims=True)) if SOUSTRAIRE_MOYENNE else M

    dt = d["dt"]
    X_brut = en_etats(roi)                      # horodatages reels : ce que BOP-DMD sait utiliser
    tt, roi_u = t, roi
    if REECHANTILLONNER_UNIFORME:
        tt = np.arange(len(t)) * dt
        plat = roi.reshape(len(t), -1)
        roi_u = np.stack([np.interp(tt, t, plat[:, k]) for k in range(plat.shape[1])],
                         axis=1).reshape(len(tt), *roi.shape[1:])

    h, w = roi.shape[1:]
    return {"X": en_etats(roi_u), "X_brut": X_brut, "shape": (h, w), "crop": (r0, r1, c0, c1),
            "t": tt, "t_brut": t, "dt": dt, "mean_image": frames.mean(axis=0)}


PREP = {s: construire_X(DATA[s]) for s in SLUGS}
for s in SLUGS:
    p = PREP[s]
    h, w = p["shape"]
    print(f"OD{replicat(s)} : ROI lignes {p['crop'][0]}–{p['crop'][1]}, colonnes {p['crop'][2]}–{p['crop'][3]}"
          f"  ->  {h} × {w} = {h * w} états × {p['X'].shape[1]} instants")

# ROI sur l'image moyenne (images encodees en PNG : sinon la figure pese des Mo)
moyennes = np.stack([PREP[s]["mean_image"].astype(np.uint8) for s in SLUGS], axis=-1)
fig = px.imshow(moyennes, facet_col=2, binary_string=True, facet_col_spacing=0.02)
for i, s in enumerate(SLUGS):
    fig.layout.annotations[i]["text"] = f"OD{replicat(s)}"
    r0, r1, c0, c1 = PREP[s]["crop"]
    fig.add_shape(type="rect", x0=c0, x1=c1, y0=r0, y1=r1, line=dict(color="#eb6834", width=2),
                  row=1, col=i + 1)
fig.update_layout(height=320, margin=dict(l=10, r=10, t=40, b=10),
                  title="ROI retenue, sur l'image moyenne")
fig.show()

OD1 : ROI lignes 229–354, colonnes 96–672  ->  31 × 72 = 2232 états × 348 instants
OD2 : ROI lignes 177–306, colonnes 96–672  ->  32 × 72 = 2304 états × 348 instants
OD3 : ROI lignes 194–321, colonnes 96–672  ->  31 × 72 = 2232 états × 348 instants


In [ ]:
# manifold_opt effectif : par variete si MANIFOLD_OPT est un dict, sinon la meme valeur
def option_de(man: str, h: int, w: int):
    if man not in MANIFOLDS_OPT_UTILE:
        return None                        # ignore par PyDMD : on ne fait pas croire le contraire
    o = MANIFOLD_OPT.get(man) if isinstance(MANIFOLD_OPT, dict) else MANIFOLD_OPT
    if man in MANIFOLDS_BLOCS and o is None:
        return (h, w)                      # blocs = une colonne A-scan (ordre Fortran)
    return o


def erreur_reconstruction(dmd, X) -> float:
    try:
        rec = np.asarray(dmd.reconstructed_data)
        return float(np.linalg.norm(rec.real - X) / np.linalg.norm(X))
    except Exception as exc:               # certaines varietes n'en fournissent pas
        print(f"  reconstruction indisponible : {type(exc).__name__}")
        return np.nan


# pi-DMD d'UNE variete sur UN replicat
def ajuster_pidmd(man: str, p: dict) -> dict:
    X, (h, w) = p["X"], p["shape"]
    n = h * w
    compute_A = COMPUTE_A or man in MANIFOLDS_A_PLEIN
    opt = option_de(man, h, w)
    if compute_A:
        go_ = n * n * 16 / 1e9
        if go_ > MEMOIRE_MAX_GO:
            raise MemoryError(f"{man} forme un A de {n}×{n} ({go_:.1f} Go > MEMOIRE_MAX_GO). "
                              f"Augmente DS_Y/DS_X ou resserre la ROI.")
    # Piege : les varietes a A plein rendent n modes, alors que les amplitudes "optimized"
    # de PyDMD supposent des modes tronques au rang SVD -> produit matriciel incompatible.
    opt_amp = False if (compute_A and OPT) else OPT

    t0 = time.perf_counter()
    dmd = PiDMD(manifold=man, manifold_opt=opt, compute_A=compute_A,
                svd_rank=SVD_RANK, tlsq_rank=TLSQ_RANK, opt=opt_amp).fit(X)
    return {"dmd": dmd, "secondes": time.perf_counter() - t0, "temps": "discret", "t": p["t"],
            "detail": f"compute_A={compute_A}, manifold_opt={opt}, opt={opt_amp}",
            "erreur": erreur_reconstruction(dmd, X)}


# Valeurs propres de depart : f0 et ses harmoniques, en paires conjuguees.
# Recopie de Astronauts/compute_pulse_from_data.py : l'importer tirerait torch pour six lignes.
def harmonic_init(rank: int, f0_hz: float, damping: float = -0.05) -> np.ndarray:
    pairs = []
    for k in range(1, rank // 2 + 1):
        w = 2j * np.pi * f0_hz * k
        pairs += [damping + w, damping - w]
    if rank % 2:
        pairs.append(damping + 0j)
    return np.array(pairs[:rank])


# BOP-DMD sur UN replicat. eigs en TEMPS CONTINU : f = Im(lambda) / 2pi.
def ajuster_bopdmd(p: dict, d: dict) -> dict:
    X = p["X_brut"] if BOP_TEMPS_REELS else p["X"]
    t = p["t_brut"] if BOP_TEMPS_REELS else p["t"]
    kw = dict(svd_rank=BOP_RANK, num_trials=BOP_NUM_TRIALS, trial_size=BOP_TRIAL_SIZE,
              eig_constraints=set(BOP_EIG_CONSTRAINTS),
              varpro_opts_dict={"maxiter": BOP_MAXITER})
    if BOP_INIT_FC and np.isfinite(d["hr_bpm"]):
        kw["init_alpha"] = harmonic_init(BOP_RANK, d["hr_bpm"] / 60.0)

    t0 = time.perf_counter()
    repli = ""
    np.random.seed(BOP_SEED)           # BOPDMD tire ses bags sur le RNG global de numpy
    with warnings.catch_warnings(record=True) as captures:
        warnings.simplefilter("always")
        try:
            bop = BOPDMD(**kw).fit(X, t)
        except Exception as exc:       # le bagging casse parfois (LinAlgError au rang eleve)
            repli = f", bagging abandonné ({type(exc).__name__})"
            kw["num_trials"] = 0
            bop = BOPDMD(**kw).fit(X, t)
        converge = not any("converge" in str(c.message) for c in captures)
    return {"dmd": bop, "secondes": time.perf_counter() - t0, "temps": "continu", "t": t,
            "detail": f"rang={BOP_RANK}, num_trials={kw['num_trials']}, init_FC={BOP_INIT_FC}, "
                      f"convergé={converge}{repli}",
            "erreur": erreur_reconstruction(bop, X)}


# (ecart-type des valeurs propres, des amplitudes) si le bagging a tourne
def ecart_types(dmd):
    try:
        e, a = dmd.eigenvalues_std, dmd.amplitudes_std
    except Exception:
        return None, None
    return (None if e is None else np.asarray(e)), (None if a is None else np.asarray(a))


# Un tableau par mode : frequence (BPM), croissance (1/s), amplitude, energie
def spectre(fit: dict, dt: float) -> pd.DataFrame:
    dmd = fit["dmd"]
    eigs = np.asarray(dmd.eigs)
    norme = np.linalg.norm(np.asarray(dmd.modes), axis=0)
    amp = np.abs(np.asarray(dmd.amplitudes))
    if fit["temps"] == "continu":          # BOP-DMD : lambda est deja un taux par seconde
        f_bpm = eigs.imag / (2 * np.pi) * 60.0
        croiss = eigs.real
        lam_disc = np.exp(eigs * dt)       # pour la figure du cercle unite
    else:
        f_bpm = np.angle(eigs) / (2 * np.pi * dt) * 60.0
        with np.errstate(divide="ignore"):
            croiss = np.log(np.abs(eigs)) / dt
        lam_disc = eigs
    tab = pd.DataFrame({
        "mode": np.arange(len(eigs)), "f_bpm": f_bpm, "abs_lambda": np.abs(lam_disc),
        "croissance_1s": croiss, "amplitude": amp, "energie": amp * norme,
        "lam_re": lam_disc.real, "lam_im": lam_disc.imag,
    })
    e_std, a_std = ecart_types(dmd)
    if e_std is not None:
        # Le bagging bouge surtout l'amortissement : Re(lambda) disperse, Im(lambda) presque pas.
        tab["f_bpm_std"] = np.abs(e_std.imag) / (2 * np.pi) * 60.0
        tab["croissance_std"] = np.abs(e_std.real)
    if a_std is not None:
        tab["energie_std"] = np.abs(a_std) * norme
    return tab


# COSTS : un BOP-DMD par fenetre glissante, puis k-means sur Im(omega).
# ATTENTION : COSTS.fit veut le temps en 2D, et omega est en temps CONTINU.
def ajuster_costs(p: dict, d: dict) -> dict:
    X = p["X_brut"] if BOP_TEMPS_REELS else p["X"]
    t = p["t_brut"] if BOP_TEMPS_REELS else p["t"]
    t0 = time.perf_counter()
    c = COSTS(svd_rank=COSTS_RANK, global_svd=COSTS_GLOBAL_SVD,
              use_last_freq=COSTS_USE_LAST_FREQ,
              pydmd_kwargs={"eig_constraints": set(BOP_EIG_CONSTRAINTS),
                            "varpro_opts_dict": {"maxiter": COSTS_MAXITER}})
    c.fit(X, t[None, :], window_length=COSTS_WINDOW, step_size=COSTS_STEP)
    c.cluster_omega(n_components=COSTS_N_BANDES, transform_method=COSTS_TRANSFORM)

    f_bpm = np.abs(np.asarray(c.omega_array).imag) / (2 * np.pi) * 60.0   # (fenetres, rang)
    amp = np.abs(np.asarray(c.amplitudes_array))
    centres = np.asarray(c.time_array).mean(axis=1)                        # (fenetres,)
    # Suivi de la bande cardiaque : par fenetre, le mode le plus proche de la FC.
    k = np.argmin(np.abs(f_bpm - d["hr_bpm"]), axis=1)
    lignes = np.arange(len(k))
    return {"costs": c, "f_bpm": f_bpm, "amp": amp, "centres": centres,
            "classes": np.asarray(c.omega_classes), "modes": np.asarray(c.modes_array),
            "suivi_f": f_bpm[lignes, k], "suivi_amp": amp[lignes, k], "suivi_k": k,
            "secondes": time.perf_counter() - t0, "n_slides": int(c.n_slides)}


# --- Ajustements ------------------------------------------------------------
FIT, SPEC, STD, COST = {}, {}, {}, {}
for s in SLUGS:
    if EST_COSTS:
        COST[s] = ajuster_costs(PREP[s], DATA[s])
        c = COST[s]
        print(f"OD{replicat(s)} · COSTS (fenetre {COSTS_WINDOW} frames = "
              f"{COSTS_WINDOW * PREP[s]['dt']:.1f} s, pas {COSTS_STEP}, rang {COSTS_RANK}) -> "
              f"{c['n_slides']} fenetres en {c['secondes']:.0f} s ; FC suivie : "
              f"mediane {fr(np.median(c['suivi_f']))}, de {fr(c['suivi_f'].min())} a "
              f"{fr(c['suivi_f'].max())} BPM (reference {fr(DATA[s]['hr_bpm'])})")
        continue
    if COMPARER_DMD_STANDARD:
        std = DMD(svd_rank=SVD_RANK, tlsq_rank=TLSQ_RANK, opt=OPT).fit(PREP[s]["X"])
        STD[s] = {"dmd": std, "erreur": erreur_reconstruction(std, PREP[s]["X"])}
    for mod in MODELES:
        FIT[(mod, s)] = (ajuster_bopdmd(PREP[s], DATA[s]) if mod == BOP
                         else ajuster_pidmd(mod, PREP[s]))
        SPEC[(mod, s)] = spectre(FIT[(mod, s)], PREP[s]["dt"])
        f = FIT[(mod, s)]
        print(f"OD{replicat(s)} · {mod:22s} ({f['detail']}) -> {len(f['dmd'].eigs)} modes "
              f"en {f['secondes']:.1f} s, erreur {fr(f['erreur'], 3)}")
    if COMPARER_DMD_STANDARD:
        print(f"OD{replicat(s)} · {'DMD standard':22s} -> erreur {fr(STD[s]['erreur'], 3)}")

c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.539429255444111e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\bopdmd.py:973: UserWarning: Initial trial of Optimized DMD failed to converge. Consider re-adjusting your variable projection parameters with the varpro_opts_dict and consider setting verbose=True.
  warnings.warn(msg)
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.4912150207251232e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.6435680249789713e+28. Consider preprocessing data, passing in augmented data
matrix,

OD1 · COSTS (fenetre 48 frames = 5.0 s, pas 12, rang 8) -> 26 fenetres en 78 s ; FC suivie : mediane 62,4, de 55,5 a 67,3 BPM (reference 60,7)


c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 1.9077092603517903e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\bopdmd.py:973: UserWarning: Initial trial of Optimized DMD failed to converge. Consider re-adjusting your variable projection parameters with the varpro_opts_dict and consider setting verbose=True.
  warnings.warn(msg)
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.118678959837055e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.61995629911478e+28. Consider preprocessing data, passing in augmented data
matrix, o

OD2 · COSTS (fenetre 48 frames = 5.0 s, pas 12, rang 8) -> 26 fenetres en 80 s ; FC suivie : mediane 60,8, de 48,1 a 72,0 BPM (reference 60,7)


c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.0435177835668254e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\bopdmd.py:973: UserWarning: Initial trial of Optimized DMD failed to converge. Consider re-adjusting your variable projection parameters with the varpro_opts_dict and consider setting verbose=True.
  warnings.warn(msg)
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.694428645031374e+28. Consider preprocessing data, passing in augmented data
matrix, or regularization methods.
  warnings.warn(
c:\Users\transformer\anaconda3\envs\pyOR\Lib\site-packages\pydmd\snapshots.py:73: UserWarning: Input data condition number 2.643522000620023e+28. Consider preprocessing data, passing in augmented data
matrix, 

OD3 · COSTS (fenetre 48 frames = 5.0 s, pas 12, rang 8) -> 26 fenetres en 79 s ; FC suivie : mediane 60,2, de 48,8 a 71,1 BPM (reference 60,7)


## 3. Les modes (`pi-DMD` et `BOP-DMD`)

En mode `COSTS`, cette section ne produit rien : ses figures sont en section 4.

Valeurs propres, puis modes spatiaux (module et phase) et modes temporels. **Une figure de modes par
modèle** (chaque variété pi-DMD, puis BOP-DMD) ; dans chacune, une colonne par réplicat et une ligne
par mode retenu (`N_MODES`, triés par `TRI_MODES`).

**Deux choses à savoir en lisant ces résultats** :

- l'erreur de reconstruction frôle 1. C'est attendu : une fois la moyenne temporelle retirée, la
  variance pixel à pixel est dominée par le bruit de speckle, qu'un modèle de rang modeste ne
  reproduit pas. Ce qu'on regarde ici, ce sont les modes, pas la reconstruction ;
- **les modes les plus énergétiques sont à 0–20 BPM**, pas à la FC (≈ 61 BPM) : c'est la dérive lente
  d'intensité et de position résiduelle. D'où `TRI_MODES = "proche_fc"`, qui va chercher le mode
  cardiaque au lieu du plus fort.

`|λ| > 1` signale un mode **instable** : sa dynamique `b λᵗ` diverge sur les ~350 pas de la vidéo et
fait exploser l'erreur de reconstruction. Les variétés `unitary` et `circulant_unitary` l'interdisent
par construction (`|λ| = 1`) ; pour BOP-DMD, l'équivalent est `croissance = Re(λ) > 0`, que montre
directement la figure croissance/fréquence qui lui est propre.

In [ ]:
def modes_retenus(cle, n_modes: int) -> np.ndarray:
    """Indices des modes montres : une seule moitie de chaque paire conjuguee."""
    _, slug = cle
    sp = SPEC[cle]
    garde = sp[sp["f_bpm"] >= -1e-9].copy()          # imag >= 0
    if TRI_MODES == "proche_fc":
        garde["cle"] = (garde["f_bpm"] - DATA[slug]["hr_bpm"]).abs()
        garde = garde.sort_values("cle")
    else:
        garde = garde.sort_values("energie", ascending=False)
    return garde["mode"].to_numpy()[:n_modes]


IDX = {cle: modes_retenus(cle, N_MODES) for cle in FIT}

lignes = []
if not MODELES:
    print(f"METHODE = {METHODE} : pas de modes stationnaires à tabuler, voir la section 4.")
for mod in MODELES:
    for s in SLUGS:
        cols = ["mode", "f_bpm", "abs_lambda", "croissance_1s", "energie"]
        cols += [c for c in ("f_bpm_std", "croissance_std") if c in SPEC[(mod, s)]]
        t = SPEC[(mod, s)].loc[IDX[(mod, s)], cols].copy()
        t.insert(0, "modèle", mod)
        t.insert(1, "réplicat", replicat(s))
        t["FC (BPM)"] = DATA[s]["hr_bpm"]
        lignes.append(t)
if lignes:
    tab = pd.concat(lignes, ignore_index=True).rename(columns={
        "f_bpm": "fréquence (BPM)", "f_bpm_std": "± (BPM)", "abs_lambda": "|λ|",
        "croissance_1s": "croissance (1/s)", "croissance_std": "± (1/s)", "energie": "énergie"})
    display(tab.round({"fréquence (BPM)": 1, "± (BPM)": 3, "|λ|": 4, "croissance (1/s)": 3,
                       "± (1/s)": 3, "énergie": 3,
                       "FC (BPM)": 1}).set_index(["modèle", "réplicat", "mode"]))

METHODE = COSTS : pas de modes stationnaires à tabuler, voir la section 4.


In [ ]:
# f_max sert aussi aux figures COSTS de la section 4.
f_max = 30 * float(np.median([d["fs"] for d in DATA.values()]))


# Plan complexe : une colonne par modele (+ le DMD standard), couleur = replicat.
# Les valeurs propres continues de BOP-DMD sont tracees comme exp(lambda dt).
def figure_plan_complexe():
    n_col = len(MODELES) + (1 if COMPARER_DMD_STANDARD else 0)
    titres = [f"{m} (exp λ dt)" if m == BOP else m for m in MODELES]
    titres += ["DMD standard"] if COMPARER_DMD_STANDARD else []
    fig = make_subplots(rows=1, cols=n_col, horizontal_spacing=0.05, subplot_titles=titres)
    theta = np.linspace(0, 2 * np.pi, 400)
    for j in range(1, n_col + 1):
        fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", showlegend=False,
                                 line=dict(color="#999", width=1, dash="dot")), row=1, col=j)
    for j, mod in enumerate(MODELES, start=1):
        for s in SLUGS:
            r, sp = replicat(s), SPEC[(mod, s)]
            fig.add_trace(go.Scatter(x=sp["lam_re"], y=sp["lam_im"], mode="markers", name=f"OD{r}",
                                     legendgroup=f"OD{r}", showlegend=(j == 1),
                                     marker=dict(color=C_REPLICAT[r], symbol=SYMB_REPLICAT[r],
                                                 size=7)), row=1, col=j)
    if COMPARER_DMD_STANDARD:
        for s in SLUGS:
            r = replicat(s)
            eigs = np.asarray(STD[s]["dmd"].eigs)
            fig.add_trace(go.Scatter(x=eigs.real, y=eigs.imag, mode="markers", showlegend=False,
                                     legendgroup=f"OD{r}",
                                     marker=dict(color=C_REPLICAT[r], symbol=SYMB_REPLICAT[r],
                                                 size=7)), row=1, col=n_col)
    fig.update_xaxes(title_text="Re λ")
    fig.update_yaxes(title_text="Im λ", row=1, col=1)
    fig.update_layout(height=380, margin=dict(l=10, r=10, t=60, b=10),
                      title="Valeurs propres dans le plan complexe (cercle unité en pointillé)",
                      legend=dict(orientation="h", y=-0.25))
    fig.show()


# Energie des modes : une colonne par replicat, couleur = modele, barres d'erreur du bagging
def figure_energie():
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=[f"OD{replicat(s)}" for s in SLUGS])
    for j, s in enumerate(SLUGS, start=1):
        for mod in MODELES:
            pos = SPEC[(mod, s)].query("f_bpm >= 0")
            err_x = dict(type="data", array=pos["f_bpm_std"]) if "f_bpm_std" in pos else None
            err_y = dict(type="data", array=pos["energie_std"]) if "energie_std" in pos else None
            fig.add_trace(go.Scatter(x=pos["f_bpm"], y=pos["energie"], mode="markers", name=mod,
                                     legendgroup=mod, showlegend=(j == 1),
                                     error_x=err_x, error_y=err_y,
                                     marker=dict(color=C_MODELE[mod], size=7, opacity=0.8)),
                          row=1, col=j)
        fc = DATA[s]["hr_bpm"]
        fig.add_vrect(x0=fc * (1 - BANDE_FC_FRAC), x1=fc * (1 + BANDE_FC_FRAC), row=1, col=j,
                      fillcolor="#eb6834", opacity=0.10, line_width=0)
        fig.add_vline(x=fc, row=1, col=j, line=dict(color="#eb6834", width=1, dash="dash"))
    fig.update_xaxes(title_text="fréquence (BPM)", range=[0, f_max])
    fig.update_yaxes(title_text="énergie |b| ‖φ‖", type="log", row=1, col=1)
    fig.update_layout(height=400, margin=dict(l=10, r=10, t=60, b=10),
                      title=f"Énergie des modes (bande orange : FC ± {int(100 * BANDE_FC_FRAC)} %)",
                      legend=dict(orientation="h", y=-0.25))
    fig.show()


# Lecture propre a BOP-DMD : croissance Re(lambda) contre frequence
def figure_bop_croissance():
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=[f"OD{replicat(s)}" for s in SLUGS])
    for j, s in enumerate(SLUGS, start=1):
        sp = SPEC[(BOP, s)].query("f_bpm >= 0")
        err_x = dict(type="data", array=sp["f_bpm_std"]) if "f_bpm_std" in sp else None
        err_y = dict(type="data", array=sp["croissance_std"]) if "croissance_std" in sp else None
        fig.add_trace(go.Scatter(x=sp["f_bpm"], y=sp["croissance_1s"], mode="markers+text",
                                 text=[f"{a:.2f}" for a in sp["energie"]], textposition="top center",
                                 textfont=dict(size=9, color="#777"), error_x=err_x, error_y=err_y,
                                 showlegend=False,
                                 marker=dict(color=C_REPLICAT[replicat(s)], size=9)), row=1, col=j)
        fc = DATA[s]["hr_bpm"]
        fig.add_vrect(x0=fc * (1 - BANDE_FC_FRAC), x1=fc * (1 + BANDE_FC_FRAC), row=1, col=j,
                      fillcolor="#eb6834", opacity=0.10, line_width=0)
        fig.add_vline(x=fc, row=1, col=j, line=dict(color="#eb6834", width=1, dash="dash"))
        fig.add_hline(y=0, row=1, col=j, line=dict(color="#999", width=1, dash="dot"))
    fig.update_xaxes(title_text="fréquence (BPM)", range=[0, f_max])
    fig.update_yaxes(title_text="croissance Re(λ) (1/s)", row=1, col=1)
    fig.update_layout(height=400, margin=dict(l=10, r=10, t=70, b=10),
                      title="BOP-DMD — croissance contre fréquence (barres : écart-type du bagging, "
                            "presque tout sur la verticale ; étiquette : énergie ; "
                            "au-dessus de 0 : le mode diverge)")
    fig.show()


if MODELES:
    figure_plan_complexe()
    figure_energie()
    if BOP in MODELES:
        figure_bop_croissance()

In [ ]:
def carte(cle, k, quoi):
    """Un mode remis en image (h, w) — ordre FORTRAN, celui de construire_X."""
    h, w = PREP[cle[1]]["shape"]
    phi = np.asarray(FIT[cle]["dmd"].modes)[:, k].reshape(h, w, order="F")
    return np.abs(phi) if quoi == "module" else np.angle(phi)


ECHELLE = {"module": "Viridis", "phase": px.colors.cyclical.Twilight}
for man in MODELES:
    for quoi in ["module"] + (["phase"] if AFFICHER_PHASE else []):
        titres = [f"OD{replicat(s)} — mode {IDX[(man, s)][i]} · "
                  f"{fr(SPEC[(man, s)].loc[IDX[(man, s)][i], 'f_bpm'])} BPM"
                  for i in range(N_MODES) for s in SLUGS]
        fig = make_subplots(rows=N_MODES, cols=3, subplot_titles=titres,
                            horizontal_spacing=0.05, vertical_spacing=0.09)
        for i in range(N_MODES):
            for j, s in enumerate(SLUGS, start=1):
                z = carte((man, s), IDX[(man, s)][i], quoi)
                kw = dict(zmin=-np.pi, zmax=np.pi) if quoi == "phase" else {}
                fig.add_trace(go.Heatmap(z=z, colorscale=ECHELLE[quoi], showscale=(i == 0 and j == 3),
                                         colorbar=dict(len=0.8 / N_MODES, y=1.0, yanchor="top"), **kw),
                              row=i + 1, col=j)
                fig.update_yaxes(autorange="reversed", showticklabels=False, row=i + 1, col=j)
                fig.update_xaxes(showticklabels=False, row=i + 1, col=j)
        fig.update_annotations(font_size=11)
        fig.update_layout(height=180 * N_MODES + 80, margin=dict(l=10, r=10, t=70, b=10),
                          title=f"Modes spatiaux · <b>{man}</b> — {quoi} "
                                f"(ROI sous-échantillonnée {DS_Y}×{DS_X})")
        fig.show()

In [ ]:
# Modes temporels : une figure par modele
for man in MODELES:
    titres = [f"OD{replicat(s)} — mode {IDX[(man, s)][i]} · "
              f"{fr(SPEC[(man, s)].loc[IDX[(man, s)][i], 'f_bpm'])} BPM · "
              f"|λ| {fr(SPEC[(man, s)].loc[IDX[(man, s)][i], 'abs_lambda'], 3)}"
              for i in range(N_MODES) for s in SLUGS]
    fig = make_subplots(rows=N_MODES, cols=3, subplot_titles=titres, shared_xaxes=True,
                        horizontal_spacing=0.05, vertical_spacing=0.07)
    for i in range(N_MODES):
        for j, s in enumerate(SLUGS, start=1):
            dyn = np.asarray(FIT[(man, s)]["dmd"].dynamics)[IDX[(man, s)][i]]
            fig.add_trace(go.Scatter(x=FIT[(man, s)]["t"], y=dyn.real, mode="lines", showlegend=False,
                                     line=dict(color=C_REPLICAT[replicat(s)], width=1.4)),
                          row=i + 1, col=j)
            fig.update_yaxes(title_text="Re b λᵗ" if j == 1 else None, row=i + 1, col=j)
    fig.update_xaxes(title_text="temps (s)", row=N_MODES, col=1)
    fig.update_annotations(font_size=11)
    fig.update_layout(height=190 * N_MODES + 80, margin=dict(l=10, r=10, t=70, b=10),
                      title=f"Modes temporels · <b>{man}</b> — Re(b_k λ_kᵗ)")
    fig.show()


# Contenu frequentiel des memes dynamiques : une ligne par modele
def figure_fft():
    fig = make_subplots(rows=len(MODELES), cols=3, shared_yaxes=True, shared_xaxes=True,
                        horizontal_spacing=0.04, vertical_spacing=0.10,
                        subplot_titles=[f"{man} — OD{replicat(s)}"
                                        for man in MODELES for s in SLUGS])
    for i, man in enumerate(MODELES, start=1):
        for j, s in enumerate(SLUGS, start=1):
            p, dyn_all = PREP[s], np.asarray(FIT[(man, s)]["dmd"].dynamics)
            freqs = np.fft.rfftfreq(dyn_all.shape[1], p["dt"]) * 60.0   # dt median si t irregulier
            for k in range(N_MODES):
                y = np.abs(np.fft.rfft(dyn_all[IDX[(man, s)][k]].real))
                fig.add_trace(go.Scatter(x=freqs, y=y / max(y.max(), 1e-12), mode="lines",
                                         name=f"mode {k + 1}", legendgroup=f"m{k}",
                                         showlegend=(i == 1 and j == 1), line=dict(width=1.3)),
                              row=i, col=j)
            fig.add_vline(x=DATA[s]["hr_bpm"], row=i, col=j,
                          line=dict(color="#eb6834", width=1, dash="dash"))
    fig.update_xaxes(title_text="fréquence (BPM)", range=[0, f_max], row=len(MODELES))
    fig.update_yaxes(title_text="|FFT| normalisé", col=1)
    fig.update_annotations(font_size=11)
    fig.update_layout(height=300 * len(MODELES) + 60, margin=dict(l=10, r=10, t=70, b=10),
                      title="Contenu fréquentiel des modes temporels (trait orange : FC du réplicat)",
                      legend=dict(orientation="h", y=-0.12))
    fig.show()


if MODELES:
    figure_fft()

## 4. COSTS — la fréquence au fil du temps

Ne s'affiche qu'avec `METHODE = "COSTS"`. Trois lectures :

1. **toutes les valeurs propres**, une par fenêtre et par mode, colorées par la bande de fréquence
   trouvée par le *k*-means — c'est la vue « spectrogramme » de la décomposition ;
2. **le suivi de la bande cardiaque** : dans chaque fenêtre, le mode le plus proche de la FC de
   référence, avec l'amplitude en épaisseur de point. C'est la courbe `f(t)` cherchée ;
3. **les modes spatiaux de ce mode suivi**, pour `COSTS_N_CARTES` fenêtres réparties sur
   l'acquisition : si la carte change de forme au cours du temps, la structure spatiale de la
   pulsation n'est pas stationnaire non plus.


In [ ]:
if EST_COSTS:
    # 1) Toutes les valeurs propres : une par (fenetre, mode), couleur = bande k-means
    fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=[f"OD{replicat(s)}" for s in SLUGS])
    for j, s in enumerate(SLUGS, start=1):
        c = COST[s]
        x = np.repeat(c["centres"], c["f_bpm"].shape[1])
        y = c["f_bpm"].ravel()
        bande = c["classes"].ravel()
        taille = 4 + 10 * (c["amp"].ravel() / max(c["amp"].max(), 1e-12)) ** 0.4
        for b in range(COSTS_N_BANDES):
            k = bande == b
            fig.add_trace(go.Scatter(x=x[k], y=y[k], mode="markers", name=f"bande {b}",
                                     legendgroup=f"b{b}", showlegend=(j == 1),
                                     marker=dict(size=taille[k], opacity=0.7,
                                                 color=px.colors.qualitative.D3[b])),
                          row=1, col=j)
        fc = DATA[s]["hr_bpm"]
        fig.add_hrect(y0=fc * (1 - BANDE_FC_FRAC), y1=fc * (1 + BANDE_FC_FRAC), row=1, col=j,
                      fillcolor="#eb6834", opacity=0.10, line_width=0)
        fig.add_hline(y=fc, row=1, col=j, line=dict(color="#eb6834", width=1, dash="dash"))
    fig.update_xaxes(title_text="temps (s)")
    fig.update_yaxes(title_text="fréquence (BPM)", range=[0, f_max], row=1, col=1)
    fig.update_layout(height=430, margin=dict(l=10, r=10, t=70, b=10),
                      title=f"COSTS — toutes les valeurs propres, fenêtre de {COSTS_WINDOW} frames "
                            f"(taille du point : amplitude ; bande orange : FC ± "
                            f"{int(100 * BANDE_FC_FRAC)} %)",
                      legend=dict(orientation="h", y=-0.2))
    fig.show()

    # 2) Suivi de la bande cardiaque + tableau recapitulatif
    fig = go.Figure()
    for s in SLUGS:
        c, r = COST[s], replicat(s)
        taille = 5 + 10 * (c["suivi_amp"] / max(c["suivi_amp"].max(), 1e-12)) ** 0.4
        fig.add_trace(go.Scatter(x=c["centres"], y=c["suivi_f"], mode="lines+markers",
                                 name=f"OD{r}", line=dict(color=C_REPLICAT[r], width=1.6),
                                 marker=dict(size=taille, color=C_REPLICAT[r])))
        fig.add_hline(y=DATA[s]["hr_bpm"], line=dict(color=C_REPLICAT[r], width=1, dash="dot"))
    fig.update_layout(height=420, margin=dict(l=10, r=10, t=70, b=10),
                      xaxis_title="temps (s)", yaxis_title="fréquence suivie (BPM)",
                      title="COSTS — la bande cardiaque fenêtre par fenêtre "
                            "(pointillés : FC de référence du réplicat)")
    fig.show()

    display(pd.DataFrame([{
        "réplicat": replicat(s), "fenêtres": COST[s]["n_slides"],
        "FC de référence": round(DATA[s]["hr_bpm"], 1),
        "médiane (BPM)": round(float(np.median(COST[s]["suivi_f"])), 1),
        "min": round(float(COST[s]["suivi_f"].min()), 1),
        "max": round(float(COST[s]["suivi_f"].max()), 1),
        "écart-type": round(float(COST[s]["suivi_f"].std()), 2),
        "secondes": round(COST[s]["secondes"]),
    } for s in SLUGS]).set_index("réplicat"))

,fenêtres,FC de référence,médiane (BPM),min,max,écart-type,secondes
réplicat,,,,,,,
1,26,60.7,62.4,55.5,67.3,3.09,78
2,26,60.7,60.8,48.1,72.0,4.47,80
3,26,60.7,60.2,48.8,71.1,4.79,79


In [ ]:
if EST_COSTS:
    # 3) Modes spatiaux du mode suivi, pour quelques fenetres reparties dans le temps
    for s in SLUGS:
        c = COST[s]
        h, w = PREP[s]["shape"]
        idx = np.unique(np.linspace(0, c["n_slides"] - 1, COSTS_N_CARTES).astype(int))
        titres = [f"t = {fr(c['centres'][k])} s · {fr(c['suivi_f'][k])} BPM" for k in idx]
        fig = make_subplots(rows=1, cols=len(idx), subplot_titles=titres, horizontal_spacing=0.04)
        for col, k in enumerate(idx, start=1):
            phi = c["modes"][k][:, c["suivi_k"][k]].reshape(h, w, order="F")
            fig.add_trace(go.Heatmap(z=np.abs(phi), colorscale="Viridis", showscale=(col == len(idx))),
                          row=1, col=col)
            fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=col)
            fig.update_xaxes(showticklabels=False, row=1, col=col)
        fig.update_annotations(font_size=11)
        fig.update_layout(height=280, margin=dict(l=10, r=10, t=70, b=10),
                          title=f"COSTS · OD{replicat(s)} — module du mode cardiaque, "
                                f"fenêtre par fenêtre")
        fig.show()